# Single Pendulum Neural Network

In [1]:
import numpy as np
import networkx as nx
import plotly.graph_objects as go

All inputs for the AI
Cart Position - where the cart is
Cart Velocity - to stop overshooting and knowing what happens to cart between each frame instead of it learning it itself
Pendulum X - Gonna have x and y instead of $\theta$ since 0 doesn't smoothly go to 359 even in radian while x and y even if they have doubles
Pendulum Y - Never gonna guess but same as X
Pendulum Angular Velocity - Momentum of the pendulum

## Neat Algorithm

In [2]:
g = 9.81
l = 1.0
m = 1.0
dt = 0.02
n = 100 # Gonna be how many pendulums I'm going to spawn

### Representing as a DAG

In [3]:
genome = {
    "nodes": {
        0: {"type": "INPUT", "act": "identity"},  # x
        1: {"type": "INPUT", "act": "identity"},  # v
        2: {"type": "INPUT", "act": "identity"},  # θ
        3: {"type": "INPUT", "act": "identity"},  # ω
        4: {"type": "OUTPUT", "act": "tanh"},  # Motor Force (-1 to 1)
    },
    "connections": {
        # Link from angle to force
        101: {"in": 2, "out": 4, "weight": 0.5, "enabled": True},
        # Link from angular velocity to force
        102: {"in": 3, "out": 4, "weight": -0.5, "enabled": True},
    },
}

In [29]:
genome = {
    "nodes": {
        0: {"type": "INPUT", "act": "identity"},  # x
        1: {"type": "INPUT", "act": "identity"},  # v
        2: {"type": "INPUT", "act": "identity"},  # θ
        3: {"type": "INPUT", "act": "identity"},  # ω
        4: {"type": "OUTPUT", "act": "tanh"},  # Motor Force
        5: {"type": "HIDDEN", "act": "identity"},  # Hidden Node
    },
    "connections": {
        # Disabled
        101: {"in": 2, "out": 4, "weight": 0.5, "enabled": False},
        103: {"in": 2, "out": 5, "weight": 1.0, "enabled": True},
        104: {"in": 5, "out": 4, "weight": 0.4, "enabled": True},
        102: {"in": 3, "out": 4, "weight": -0.5, "enabled": True},
        105: {"in": 1, "out": 5, "weight": -0.2, "enabled": True},
    },
}

In [12]:
def graph_dag(genome):
    G = nx.DiGraph()

    for node_id, data in genome["nodes"].items():
        G.add_node(node_id, type=data["type"], act=data["act"]) # With info to be displayed

    for conn in genome["connections"].values():
        if conn["enabled"]:
            G.add_edge(conn["in"], conn["out"], weight=conn["weight"])

    for node_id, data in genome["nodes"].items():
        if data["type"] == "INPUT":
            G.nodes[node_id]["layer"] = 0
        elif data["type"] == "OUTPUT":
            G.nodes[node_id]["layer"] = 2
        else:
            G.nodes[node_id]["layer"] = 1

    pos = nx.multipartite_layout(G, subset_key="layer") # This is going to be the positions of these

    edge_traces = []
    for edge in G.edges(data=True):
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        weight = edge[2]["weight"]

        color = "rgba(50, 100, 250, 0.8)" if weight > 0 else "rgba(250, 50, 50, 0.8)"

        trace = go.Scatter(
            x=[x0, x1, None],
            y=[y0, y1, None],
            line=dict(width=abs(weight) * 5, color=color),
            hoverinfo="none",
            mode="lines",
        )
        edge_traces.append(trace)

    node_x, node_y, node_hover = [], [], []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_hover.append(
            f"Node {node}<br>Type: {G.nodes[node]['type']}<br>Act: {G.nodes[node]['act']}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=[f"N{n}" for n in G.nodes()],
        textposition="top center",
        hoverinfo="text",
        hovertext=node_hover,
        marker=dict(
            size=25,
            color=[
                "#636EFA" if G.nodes[n]["type"] == "INPUT" else "#EF553B"
                for n in G.nodes()
            ],
            line_width=2,
        ),
    )

    fig = go.Figure(
        data=edge_traces + [node_trace],
        layout=go.Layout(
            title="Evolved Pendulum Controller (NEAT)",
            showlegend=False,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor="white",
        ),
    )
    fig.show()

In [13]:
graph_dag(genome)

### Solve DAG

In [60]:
inputs = [0.5, 0.1, 0.2, 0.3]  # Initial values for [x, v, θ, ω]

#### Kahns

So far I know we can do Kahn's Algorithm or use DFS. Gonna do Kahn's for now.

In [31]:
# Get Topological Order
nodes = genome["nodes"]
connections = genome["connections"]

in_degree = {node_id: 0 for node_id in nodes}
for conn in connections.values():
    # Basically for every connection we add to the in_degree of every node or how many connection go into node
    if conn["enabled"]:
        in_degree[conn["out"]] += 1

# So we do a queue where if the degree is 0 then no connection go into node like the INPUT nodes
queue = [node_id for node_id, degree in in_degree.items() if degree == 0]
topo_order = []

while queue:
    current = queue.pop(0)
    topo_order.append(current)

    # For every connection going out of the current node, we reduce the in_degree
    for conn in connections.values():
        # So in basically means starting so if it starts from the current node then we basically say it's use up and the connection is broken
        # Since once we have the value and let's say next item has 0 connection then this is the only connection but if it has 1 then we wait till that one is also broken
        if conn["enabled"] and conn["in"] == current:
            in_degree[conn["out"]] -= 1
            # If it's 0 then we add it to the queue
            if in_degree[conn["out"]] == 0:
                queue.append(conn["out"])

print("Topological Order:", topo_order)

Topological Order: [0, 1, 2, 3, 5, 4]


In [32]:
genome["connections"]

{101: {'in': 2, 'out': 4, 'weight': 0.5, 'enabled': False},
 103: {'in': 2, 'out': 5, 'weight': 1.0, 'enabled': True},
 104: {'in': 5, 'out': 4, 'weight': 0.4, 'enabled': True},
 102: {'in': 3, 'out': 4, 'weight': -0.5, 'enabled': True},
 105: {'in': 1, 'out': 5, 'weight': -0.2, 'enabled': True}}

In [ ]:
node_values = {node_id: 0.0 for node_id in genome["nodes"]}

for i, val in enumerate(inputs):
    node_values[i] = val

for node_id in topo_order:
    node_data = genome["nodes"][node_id]

    # if node_data["type"] != "INPUT":
    #     raw_val = node_values[node_id]
    #     if node_data["act"] == "tanh":
    #         node_values[node_id] = np.tanh(raw_val) # When we do tanh all connection into are already done so tanh is done at the end not inbetween
    #     elif node_data["act"] == "identity":
    #         node_values[node_id] = raw_val

    # Propogate forward
    for conn in genome["connections"].values():
        if conn["enabled"] and conn["in"] == node_id: # So starts from this node
            node_values[conn["out"]] += node_values[node_id] * conn["weight"] # We add to the output node

print("Motor Activation:", node_values[4])

Motor Activation: -0.078


In [ ]:
# Hide tanh for now t
node_values

{0: 0.5, 1: 0.1, 2: 0.2, 3: 0.3, 4: -0.078, 5: 0.18}

#### Matrix

In [ ]:
dag_matrix = np.zeros((len(genome["nodes"]), len(genome["nodes"])))
dag_sol = np.zeros(len(genome["nodes"]))

for i in range(len(genome["nodes"])):
    if genome["nodes"][i]["type"] == "INPUT":
        dag_matrix[i, i] = 1.0  # Identity for input nodes
        dag_sol[i] = inputs[i]
    else:
        dag_matrix[i, i] = -1.0
        dag_sol[i] = 0.0

for conn in genome["connections"].values():
    if conn["enabled"]:
        dag_matrix[conn["out"], conn["in"]] = conn["weight"]

print("DAG Adjacency Matrix:\n", dag_matrix)
print("DAG Solution Matrix:\n", dag_sol)

node_values = np.linalg.solve(dag_matrix, dag_sol)
print(node_values)

DAG Adjacency Matrix:
 [[ 1.   0.   0.   0.   0.   0. ]
 [ 0.   1.   0.   0.   0.   0. ]
 [ 0.   0.   1.   0.   0.   0. ]
 [ 0.   0.   0.   1.   0.   0. ]
 [ 0.   0.   0.  -0.5 -1.   0.4]
 [ 0.  -0.2  1.   0.   0.  -1. ]]
DAG Solution Matrix:
 [0.5 0.1 0.2 0.3 0.  0. ]
[ 0.5    0.1    0.2    0.3   -0.078  0.18 ]


It works without tanh we could apply at the end but if we want hidden nodes to have tanh then it might not work. Don't think it's worth it since unlike a neural network where every layers creates a full matrix this creates a sparse matrix. And if we have nodes that skip layers like we have for node 4 here where 3 in layers 1 and 5 from layer 2 connects to it. We need a bigger matrix with all past layers for every next layers we go to. But if we need to parallize we could do a tensor or a bigger 2D array but we now need to fit the matrix into the biggest matrix and have unncescary memory and even more sparse. So gonna go with the loop for now then try to speed up. Also could try to do matrix mult at the end since it might be faster and I could update more.

## To Do

DFS

Try to pad matrix based of the highest rank to solve as even though it is sparse the pendulum is smaller.